# Error analysis

Which masses does the model miss, and where does it see masses that are not there?
The evaluation report gives the rates; this notebook looks at the individual masses
and false regions behind them, from the per-image details that
`mammo-seg evaluate --details` writes.

It answers four questions, at one post-processing setting:

1. **Missed masses:** found or missed by size, density, BI-RADS and view.
2. **The other view:** is a mass missed in one view found in the other view of the
   same breast?
3. **False regions:** on which images, how large, where in the breast, and whether
   they lie on an annotated finding that is not a mass (calcifications, clusters,
   distortions and so on).
4. **The worst images:** the lowest Dice with a mass, and the most false area
   without one.

It needs the details file and the local `data/processed/inbreast/` (manifest and
preprocessing record), but no model and no GPU. Only scores, sizes, positions and
image ids are shown, no images: look an image up by its id in the review notebooks
if needed. Outputs are stripped before commit (`nbstripout`).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.path import Path as Polygon

from mammo_seg.data.manifest import read_manifest
from mammo_seg.data.paths import INBREAST_PROCESSED_DIR
from mammo_seg.data.preprocess_dataset import read_record
from mammo_seg.data.preprocessing import preprocess_points
from mammo_seg.data.schema import LesionType
from mammo_seg.data.strata import DEFAULT_DENSE_ACR, DEFAULT_SMALL_MASS_MM2
from mammo_seg.engine.evaluate import NEGATIVE_KINDS


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("pyproject.toml not found above " + str(start))


ROOT = find_repo_root(Path.cwd().resolve())
PROCESSED = ROOT / INBREAST_PROCESSED_DIR
DETAILS = (
    ROOT
    / "results"
    / "2026-10-24_whole_vs_patches"
    / "whole"
    / "evaluation_validation_details.json"
)
THRESHOLD = 0.4  # the setting to analyse; it must be one of the details file
MIN_AREA_MM2 = 10.0
NEAR_MM = 5.0  # a false region this close to an annotated finding is on it
CHEST_WALL = 0.1  # the strip next to the chest wall, as a fraction of the width
SIZE_BINS_MM2 = [0, 50, 100, 200, 500, np.inf]
BLUE, ORANGE = "#2a78d6", "#eb6834"  # the first two colours of the palette

## The details

The details file holds every validation image with its results under every
setting the evaluation tried. Only the chosen setting is used below. The manifest
gives each image its breast side and its annotations, and the preprocessing record
the geometry that maps the annotations onto the preprocessed image, where the
evaluation measured the regions.

In [ ]:
details = json.loads(DETAILS.read_text())
manifest = read_manifest(PROCESSED / "manifest.json")
record = read_record(PROCESSED / "preprocessed" / "preprocessing.json")

setting = {"threshold": THRESHOLD, "min_area_mm2": MIN_AREA_MM2}
if setting not in details["settings"]:
    raise ValueError(f"{setting} is not in {DETAILS.name}: {details['settings']}")
SETTING = details["settings"].index(setting)
records = {image.file_id: image for image in manifest.images}
unknown = {i["file_id"] for i in details["images"]} - (
    records.keys() & record.images.keys()
)
if unknown:
    raise ValueError(f"{len(unknown)} images of the details are not in {PROCESSED}")

print(f"{details['checkpoint']} on the {details['on']} images")
print(f"inference per fold: {details['inference']}")
print(f"{len(details['images'])} images at {setting}")
print(f"a mass is found at IoU >= {details['hit_iou']}")

## Images, masses and false regions

Three tables: one row per image, per mass and per false region. A false region is a
predicted region that overlaps no mass enough to find it; it can be on an image
with a mass too.

Positions are the region centres in the preprocessed image, as fractions of its
height (`rel_row`, 0 at the top) and width (`rel_col`). Every breast is flipped so
the chest wall is on the left: `rel_col` near 0 is the chest wall, near 1 the skin
or the nipple.

In [ ]:
def kind_of(image: dict) -> str:
    """``with a mass``, or the kind of an image without a mass, as in the report."""
    if image["has_mass"]:
        return "with a mass"
    return next(k for k, birads in NEGATIVE_KINDS.items() if image["birads"] in birads)


image_rows, mass_rows, false_rows = [], [], []
for image in details["images"]:
    result = image["results"][SETTING]
    rows, cols = record.images[image["file_id"]].shape
    base = {
        "fold": image["fold"],
        "file_id": image["file_id"],
        "patient_id": image["patient_id"],
        "breast": records[image["file_id"]].laterality,
        "view": image["view"],
        "birads": image["birads"],
        "acr": image["acr"],
        "kind": kind_of(image),
    }
    image_rows.append(
        base
        | {
            "dice": result["dice"],
            "masses": len(result["masses"]),
            "found": sum(m["found"] for m in result["masses"]),
            "false_regions": len(result["false_regions"]),
            "false_mm2": sum(f["area_mm2"] for f in result["false_regions"]),
        }
    )
    for found_in, region_rows in (("masses", mass_rows), ("false_regions", false_rows)):
        for region in result[found_in]:
            region_rows.append(
                base
                | region
                | {"rel_row": region["row"] / rows, "rel_col": region["col"] / cols}
            )

images = pd.DataFrame(image_rows)
masses = pd.DataFrame(mass_rows)
falses = pd.DataFrame(false_rows)
for frame in (images, masses, falses):
    frame["acr"] = frame.acr.astype("Int64")  # missing for a few images
masses["size"] = pd.cut(masses.area_mm2, SIZE_BINS_MM2, right=False)
masses["small"] = masses.area_mm2 < DEFAULT_SMALL_MASS_MM2
masses["dense"] = masses.acr >= DEFAULT_DENSE_ACR
print(
    f"{len(images)} images, {len(masses)} masses ({int(masses.found.sum())} found), "
    f"{len(falses)} false regions"
)

### Checking the geometry

The false regions are compared with the annotations below, which needs the
annotations mapped onto the preprocessed images. To check that mapping, every
found mass must lie on one of the mass polygons of its image: its centre inside the
polygon, or within `NEAR_MM`. The evaluation's centres are pixel indices, so they
are moved by half a pixel to the continuous coordinates of the polygons.

In [ ]:
def distance_mm(region: pd.Series, lesion_type: LesionType | None) -> tuple[float, str]:
    """The distance in mm from the centre of ``region`` to the nearest annotation
    of ``lesion_type`` (any type but a mass when None) of its image, with that
    annotation's type; 0 inside a polygon, inf without one."""
    params = record.images[region.file_id]
    centre = (region.col + 0.5, region.row + 0.5)  # (x, y), continuous
    mm = np.sqrt(region.pixel_area_mm2)
    nearest = (np.inf, "")
    for roi in records[region.file_id].rois:
        if (roi.lesion_type is LesionType.MASS) != (lesion_type is LesionType.MASS):
            continue
        points = preprocess_points(roi.points, params)
        if len(points) >= 3 and Polygon(points).contains_point(centre):
            return 0.0, roi.lesion_type.value
        gap = float(np.hypot(*(points - centre).T).min()) * mm
        nearest = min(nearest, (gap, roi.lesion_type.value))
    return nearest


pixel_areas = {i["file_id"]: i["pixel_area_mm2"] for i in details["images"]}
masses["pixel_area_mm2"] = masses.file_id.map(pixel_areas)
falses["pixel_area_mm2"] = falses.file_id.map(pixel_areas)
found = masses[masses.found]
off = [
    d
    for d in (distance_mm(m, LesionType.MASS)[0] for _, m in found.iterrows())
    if d > NEAR_MM
]
if off:
    raise ValueError(
        f"{len(off)} found masses are off their polygons: the geometry is wrong"
    )
print(f"all {len(found)} found masses lie on their polygons")

## 1. Missed masses

The share of masses found, by size. A missed mass is one the model predicts
(almost) nothing on: its IoU with the prediction is shown in the second plot, on a
log scale of the mass area. A mass found with a low IoU would show that the model
sees it but outlines it badly; a mass at IoU 0 is not seen at all.

In [ ]:
def found_share(frame: pd.DataFrame, by: str | list[str]) -> pd.DataFrame:
    return (
        frame.groupby(by, observed=False)
        .found.agg(masses="size", found="sum")
        .assign(share=lambda t: (t.found / t.masses).round(3))
    )


by_size = found_share(masses, "size")
display(by_size)

fig, (bars, scatter) = plt.subplots(1, 2, figsize=(12, 4))
labels = [f"{b.left:g}–{b.right:g}" for b in by_size.index]
bars.bar(labels, by_size.share, color=BLUE)
for x, (share, n) in enumerate(zip(by_size.share, by_size.masses, strict=True)):
    bars.text(
        x, share + 0.02, f"{share:.2f}\nn={n}", ha="center", va="bottom", fontsize=9
    )
bars.set(
    ylim=(0, 1.15),
    xlabel="mass area (mm²)",
    ylabel="share found",
    title="Masses found, by size",
)
for is_found, colour, label in ((True, BLUE, "found"), (False, ORANGE, "missed")):
    group = masses[masses.found == is_found]
    scatter.scatter(group.area_mm2, group.iou, s=24, color=colour, label=label)
scatter.axvline(DEFAULT_SMALL_MASS_MM2, color="grey", linestyle="--", linewidth=1)
scatter.axhline(details["hit_iou"], color="grey", linestyle=":", linewidth=1)
scatter.set(
    xscale="log",
    xlabel="mass area (mm², log)",
    ylabel="IoU with the prediction",
    title="IoU of every mass",
)
scatter.legend()
for ax in (bars, scatter):
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

missed = masses[~masses.found]
print(
    f"{len(missed)} missed masses; their IoU: max {missed.iou.max():.3f}, "
    f"{(missed.iou == 0).sum()} at exactly 0"
)

### How close did the missed masses come?

The peak of a mass is the highest predicted probability inside it, whatever the
threshold. It separates two kinds of miss that need different fixes:

- **a weak response**, a peak a little under the threshold: the model half sees
  the mass, and more weight on small masses in the loss, or a lower threshold for
  them, could find it
- **no response**, a peak near 0 like the tissue around it: the model does not see
  the mass, and the input (its resolution) is the place to look

A missed mass can also peak above the threshold: its region was then removed as too
small, or spilled so far over the mass that its IoU stayed under the hit level.
Peaks need a details file written with the code of the evaluation that records
them; an older file has none, and this part is skipped.

In [ ]:
HAS_PEAKS = "peak" in masses
if not HAS_PEAKS:
    print(
        f"{DETAILS.name} has no peaks: write it again with mammo-seg evaluate --details"
    )
else:
    fig, ax = plt.subplots(figsize=(7, 4))
    for is_found, colour, label in ((True, BLUE, "found"), (False, ORANGE, "missed")):
        group = masses[masses.found == is_found]
        ax.scatter(group.area_mm2, group.peak, s=24, color=colour, label=label)
    ax.axhline(THRESHOLD, color="grey", linestyle=":", linewidth=1)
    ax.axvline(DEFAULT_SMALL_MASS_MM2, color="grey", linestyle="--", linewidth=1)
    ax.set(
        xscale="log",
        ylim=(0, 1.02),
        xlabel="mass area (mm², log)",
        ylabel="highest probability inside",
        title="The peak of every mass",
    )
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

    response = pd.cut(
        missed.peak,
        [0, 0.05, 0.2, THRESHOLD, np.inf],
        right=False,
        labels=[
            "none (< 0.05)",
            "faint (0.05-0.2)",
            f"weak (0.2-{THRESHOLD})",
            "above the threshold",
        ],
    )
    display(
        missed.groupby([response, "small"], observed=False)
        .size()
        .unstack(fill_value=0)
        .rename(columns={True: "small", False: "large"})
        .rename_axis(index="response of the missed masses", columns=None)
    )

### Size, density, BI-RADS and view

Small masses and dense breasts may go together, so size and density are shown
together first. Each table has the number of masses, those found and the share.

In [ ]:
display(found_share(masses, ["small", "dense"]))
display(found_share(masses, "acr"))
display(found_share(masses, "birads"))
display(found_share(masses, "view"))

## 2. The other view

A radiologist reads both views of a breast, so a mass missed in one view but found
in the other is less of a miss. Every view of a patient is in the same fold, so
both views of a breast are scored by the same model.

Per breast with a mass in both views: is a mass found in the CC view, in the MLO
view, in both or in neither? A breast with several masses counts as found in a view
if any of its masses is; matching single masses across views would need their
positions in 3-D.

In [ ]:
per_view = masses.groupby(["patient_id", "breast", "view"]).found.any().unstack("view")
both_views = per_view.dropna().astype(bool)
outcome = np.select(
    [both_views.CC & both_views.MLO, both_views.CC, both_views.MLO],
    ["both views", "CC only", "MLO only"],
    "neither",
)
display(pd.Series(outcome).value_counts().rename("breasts").to_frame())
print(f"{len(per_view) - len(both_views)} breasts have a mass in one view only")

missed_breasts = missed.merge(
    both_views.reset_index(), on=["patient_id", "breast"], how="left"
)
other_view = np.where(
    missed_breasts.view == "CC", missed_breasts.MLO, missed_breasts.CC
)
rescued = int(pd.Series(other_view).eq(True).sum())
print(
    f"of {len(missed)} missed masses, {rescued} are in a breast whose other view "
    "has a mass found"
)

## 3. False regions

How many false regions there are on each kind of image, how large they are and
whether they lie on an annotated finding that is not a mass: their centre inside
its polygon or within `NEAR_MM` of it. Calcifications are often single points, so
the distance matters for them.

In [ ]:
nearest = [distance_mm(f, None) for _, f in falses.iterrows()]
falses["finding_mm"] = [d for d, _ in nearest]
falses["finding"] = [kind if d <= NEAR_MM else "none" for d, kind in nearest]
falses["chest_wall"] = falses.rel_col < CHEST_WALL

per_kind = images.groupby("kind").agg(
    images=("file_id", "size"),
    with_false_regions=("false_regions", lambda n: int((n > 0).sum())),
    false_regions=("false_regions", "sum"),
)
display(per_kind)
display(falses.finding.value_counts().rename("false regions").to_frame())
display(falses.area_mm2.describe().round(1).to_frame("false region area (mm²)"))
if HAS_PEAKS:  # how confident the false regions are: a higher threshold or not
    display(falses.peak.describe().round(3).to_frame("false region peak"))

### Where the false regions are

The centres of the false regions in each view, in the frame of the preprocessed
image (chest wall on the left, top at the top), with the size of a dot following
the region's area. In MLO views the pectoral muscle is the bright triangle in the
upper left, along the chest wall. The dashed line is the `CHEST_WALL` strip.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 6), sharey=True)
for ax, view in zip(axes, ("CC", "MLO"), strict=True):
    group = falses[falses.view == view]
    for on_finding, colour, label in (
        (False, BLUE, "not on a finding"),
        (True, ORANGE, "on a finding"),
    ):
        part = group[(group.finding != "none") == on_finding]
        ax.scatter(
            part.rel_col,
            part.rel_row,
            s=8 + part.area_mm2 / 3,
            color=colour,
            alpha=0.8,
            label=label,
            edgecolor="white",
            linewidth=1,
        )
    ax.axvline(CHEST_WALL, color="grey", linestyle="--", linewidth=1)
    ax.set(
        xlim=(0, 1),
        ylim=(1, 0),
        xlabel="chest wall → skin (fraction of width)",
        title=f"{view}: {len(group)} false regions, "
        f"{int(group.chest_wall.sum())} at the chest wall",
    )
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("top → bottom (fraction of height)")
axes[0].legend(loc="lower right")
plt.tight_layout()
plt.show()

display(falses.groupby(["view", "chest_wall"]).size().unstack(fill_value=0))

## 4. The worst images

The images with a mass with the lowest Dice, and the images with the largest false
area. The ids are for looking the images up, not for showing them here.

In [ ]:
columns = [
    "fold",
    "file_id",
    "view",
    "birads",
    "acr",
    "dice",
    "masses",
    "found",
    "false_regions",
    "false_mm2",
]
largest = masses.groupby("file_id").area_mm2.max().rename("largest_mass_mm2")
worst = images[images.kind == "with a mass"].join(largest, on="file_id")
display(worst.nsmallest(10, "dice")[[*columns, "largest_mass_mm2"]].round(3))
display(images.nlargest(10, "false_mm2")[[*columns, "kind"]].round(3))

## Reading the results

Questions to take to the next experiment:

- **Is size the problem, or density?** If small masses are missed in fatty breasts
  too, the resolution (a mass of 50 mm² is a few hundred pixels after
  preprocessing) or the loss is the likely cause, not the tissue.
- **Is a missed mass seen at all?** The peaks tell: if most missed masses have a
  weak response, start with the loss (more weight on small masses) or the
  threshold; if most have none, start with the input resolution.
- **Does the other view rescue the misses?** If most missed masses are in breasts
  found in the other view, a breast-level score is closer to how the output would
  be read than the per-image one.
- **Are the false regions anatomy or findings?** Regions at the chest wall (the
  pectoral muscle in MLO) point to anatomy the model has not learned to ignore;
  regions on calcifications or other findings are look-alikes the annotations do
  not count as masses.